# Sprint 3 (World Stock Prices): First Baseline, Time Leakage, and Metric Selection

This notebook is designed for your project type (**time series and daily stock prices**). It does not build the final solution for you. Instead, it explains **what** each ML concept and tool actually is, **why** we use it, **what breaks** if you skip it, and **how** to construct your first honest **Baseline**.

### Where We Are in the Project Roadmap
*   Sprint 1: Idea, repository setup, and `README.md` (Done)
*   Sprint 2: Data collection and EDA *(if you are still finishing this, focus on wrapping it up today)*
*   **Sprint 3 (TODAY): Chronological train/test split without Data Leakage + first simple Baseline + metric selection**
*   Upcoming sessions: Time-series cross-validation (`TimeSeriesSplit`) and error analysis -> Financial feature engineering and stronger models -> Refactoring into `src/train.py` -> Building `app/app.py` and deployment.

### Why Start With a Baseline Instead of Complex Models?
A **Baseline** is the simplest reasonable model you can build quickly to establish a **starting benchmark score**. We do not jump straight into deep learning or hyperparameter tuning because:
1.  Without a simple benchmark, you cannot tell whether a complex model is actually learning or just overfitting noise.
2.  A fast baseline exposes data bugs (bad dates, missing values, Data Leakage) on day one instead of week four.

### Skill Levels in This Notebook
*   **START:** A minimal 4-line model (no `Pipeline`) so everyone understands what `.fit()` and `.predict()` actually do.
*   **BASIC:** The core Sprint 3 deliverable (chronological split, `Pipeline`, Naive Baseline vs. 2 simple models, and metric selection).
*   **PRO:** Deeper error inspection and next-step feature ideas.

In [ ]:
# Core libraries for data manipulation, plotting, and basic machine learning
import warnings
import numpy as np                  # Fast numerical arrays and math operations
import pandas as pd                 # Tabular data (DataFrames) and time-series utilities
import matplotlib.pyplot as plt     # Base plotting library
import seaborn as sns               # Clean statistical plots built on top of matplotlib

# Scikit-learn tools for leak-free preprocessing and baseline modeling
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

sns.set_theme(style="whitegrid", palette="muted")
warnings.filterwarnings('ignore')

## Step 0: Data Status — Static CSV vs. Live API Snapshot

**If you are still setting up your dataset (Sprint 2), start here:**
*   **Option A (Static CSV from Kaggle):** Put the *World Stock Prices* CSV in `data/raw/`. Check in EDA that `Date` is converted to a true datetime type (`pd.to_datetime`), because sorting dates stored as plain text (e.g., `'02-01-2025'` vs `'10-01-2024'`) puts them in alphabetical order instead of chronological order.
*   **Option B (Live Daily Updates via API):** Instead of fragile HTML scraping, you can use the free `yfinance` library (`pip install yfinance`) to download daily Yahoo Finance quotes. **Why save a local copy to `data/raw/` anyway?** Because live API data changes every day. During model development you want a fixed dataset so that when your score changes, you know it was your code — not new market days.

*Below, we demonstrate every concept on a small synthetic dataset of 2 fictional companies (`CompanyA`, `CompanyB`). At the end, you will build the real version on your own CSV.*

In [ ]:
# =====================================================================
# REFERENCE: Loading your real data in notebooks/02_baseline.ipynb
#   Option A (Kaggle CSV):   df = pd.read_csv('../data/raw/your_file.csv')
#   Option B (yfinance API): pip install yfinance -> yf.download('AAPL', start='2022-01-01')
# =====================================================================

# --- EDUCATIONAL DEMO DATASET (2 fictional companies, 250 business days) ---
np.random.seed(42)  # Fixes random number generation so every run produces the same demo data
dates = pd.bdate_range('2025-01-01', periods=250)  # Business days only (Mon-Fri, like stock markets)
rows = []
for name, start, vol in [('CompanyA', 100.0, 0.015), ('CompanyB', 250.0, 0.025)]:
    daily_ret = np.random.normal(0.0004, vol, len(dates))
    close = start * np.cumprod(1 + daily_ret)
    volume = np.random.randint(1_000_000, 10_000_000, len(dates))
    for d, c, v in zip(dates, close, volume):
        rows.append({'Date': d, 'Company': name, 'Close': round(c, 2), 'Volume': v})

df_demo = pd.DataFrame(rows)
display(df_demo.head())

## Step 1: What Is Train/Test Splitting, What Is Data Leakage, and Why Stocks Need Special Care

### What is a Train/Test Split and why do we need it?
Imagine studying for an exam using 100 practice questions, and then getting those exact same 100 questions on the real exam. Scoring 100% would not prove you understood the subject — only that you memorized the answers. In ML, memorizing the training data is called **overfitting**. To measure real-world performance (**generalization**), we split our data into two completely separate sets:
*   **Training set (`train`):** The textbook the model learns patterns from (typically ~80% of the data).
*   **Test set (`test`):** The locked final exam the model never sees until evaluation (typically ~20%).

### What is Data Leakage?
**Data Leakage** happens whenever information from the test set or from the future accidentally enters the training process. It creates fake, overly optimistic scores in your notebook that collapse in a real app. In stock datasets, there are **three classic traps**:

1.  **Time Leakage (Why we NEVER split stock data randomly):**
    A standard `train_test_split(..., shuffle=True)` shuffles rows randomly. That lets the model train on Monday and Wednesday to 'predict' Tuesday — effectively peeking into the future. We must always split **chronologically**: older dates in `train`, newer dates in `test`.
2.  **Cross-Entity Leakage (Why we use `groupby('Company')` before `.shift()`):**
    A **lag feature** (`.shift(1)`) copies a value from the previous row (yesterday) into the current row. Because your CSV stacks multiple companies in one table, a bare `df['Close'].shift(1)` copies the last day of `CompanyA` into the first day of `CompanyB`. Grouping by company first keeps each company's timeline isolated.
3.  **Non-Stationarity (Why we predict `% return` instead of raw dollar `Close`):**
    Stock prices drift upward over years (e.g., $100-$150 in `train`, then $180 in `test`). Decision trees (`RandomForest`, `XGBoost`) work by splitting training values into ranges and returning the average training value in each leaf — so **trees cannot predict numbers outside the range they saw during training** (they hit a flat ceiling at $150). A **percentage return** — `(Close_tomorrow - Close_today) / Close_today` — stays centered near `0%` regardless of whether a share costs $20 or $500. You can always convert a predicted return back to dollars with `Today_Price * (1 + predicted_return)`.

In [ ]:
# 1. Sort by company and date so consecutive rows are truly consecutive trading days
df_demo = df_demo.sort_values(['Company', 'Date']).reset_index(drop=True)

# 2. Compute backward-looking features INSIDE each company using groupby()
# pct_change(k) computes (Close_t - Close_{t-k}) / Close_{t-k} -> relative change over the past k days
df_demo['return_today'] = df_demo.groupby('Company')['Close'].pct_change(1)  # 1-day momentum
df_demo['return_5d']    = df_demo.groupby('Company')['Close'].pct_change(5)  # 1-week (5 trading days) trend

# 3. Create the target (y): tomorrow's percentage return
# shift(-1) pulls the NEXT row's value up into today's row so today's features predict tomorrow's outcome
df_demo['target_tomorrow_return'] = df_demo.groupby('Company')['return_today'].shift(-1)

# Remove boundary rows where pct_change(5) or shift(-1) produced NaN (no history or no known tomorrow)
df_clean = df_demo.dropna().copy()

# 4. CHRONOLOGICAL SPLIT using the 80th percentile date (quantile(0.8))
# Everything on or before split_date is the past (train); everything after is the future (test)
split_date = df_clean['Date'].quantile(0.8)
train_df = df_clean[df_clean['Date'] <= split_date]
test_df  = df_clean[df_clean['Date'] > split_date]

print(f"Train period (past):   {train_df['Date'].min().date()} -> {train_df['Date'].max().date()} ({len(train_df)} rows)")
print(f"Test period  (future): {test_df['Date'].min().date()} -> {test_df['Date'].max().date()} ({len(test_df)} rows)")

## Step 2 (START -> BASIC): How ML Learning Works, and Why We Use a `Pipeline`

### Level START: Understanding `.fit()` and `.predict()` in 4 lines
Before using `Pipeline`, look at what happens under the hood when training a model on plain numbers:
*   **What is `Ridge`?** It is **Linear Regression with L2 regularization**. It tries to predict the target as a weighted sum of features (`w1 * return_today + w2 * return_5d + bias`). Regularization (`alpha`) gently shrinks extreme weights so the model does not overreact to noisy days.
*   **What does `.fit(X, y)` do?** It adjusts the weights (`w1`, `w2`, `bias`) on `train_df` to minimize prediction errors.
*   **What does `.predict(X)` do?** It plugs unseen `test_df` values into those learned weights to produce forecasts.

In [ ]:
# --- LEVEL START: Minimal ML in 4 lines (numeric columns only, no Pipeline) ---
simple_cols = ['return_today', 'return_5d']

model_simple = Ridge()                                                       # 1. Initialize linear model
model_simple.fit(train_df[simple_cols], train_df['target_tomorrow_return'])  # 2. Learn weights on past data
preds_simple = model_simple.predict(test_df[simple_cols])                    # 3. Forecast unseen future dates

mae_simple = mean_absolute_error(test_df['target_tomorrow_return'], preds_simple)
print(f"Simple model (2 features) - MAE: {mae_simple * 100:.3f}% per day")

### Level BASIC: What Each Preprocessing Step Does and Why We Use `Pipeline`
Real datasets contain missing values, mixed numeric scales, and text categories (`Company`). If we feed `Volume` (millions) alongside `return_today` (`0.01`) straight into `Ridge`, the huge scale difference distorts regularization. Here is what each scikit-learn building block does:
*   **`SimpleImputer(strategy='median')`:** Replaces missing numeric values (`NaN`) with the column median from the training set (median is safer than mean because a single extreme outlier cannot distort it).
*   **`StandardScaler()`:** Rescales each numeric column to have mean `0` and standard deviation `1` (`(x - mean) / std`), so every feature is on a fair footing.
*   **`OneHotEncoder(handle_unknown='ignore')`:** Turns text categories (e.g., `'CompanyA'`, `'CompanyB'`) into binary `0/1` columns because math equations cannot multiply strings. `handle_unknown='ignore'` prevents crashes if a new unseen company appears later in production.
*   **`ColumnTransformer`:** Routes numeric columns to the numeric steps and categorical columns to the categorical steps, then glues the results back together.
*   **`Pipeline`:** Chains `preprocessor -> model` into a single object. **Why is this critical?** When you call `pipe.fit(X_train, y_train)`, the imputer, scaler, and encoder learn statistics **strictly from `X_train`**. When you call `pipe.predict(X_test)`, they reuse those saved `X_train` statistics without peeking at `X_test` (zero **Preprocessing Leakage**).

### Why Compare Two Different Model Families Against a Naive Baseline?
*   **0. Naive / Random Walk Baseline:** Always predicts `0.0%` change (*'tomorrow's price equals today's price'*). Without beating this, an ML model adds zero value.
*   **1. `Ridge` (Linear Model):** Fast, low-variance baseline that captures straight-line relationships.
*   **2. `RandomForestRegressor` (Tree Ensemble):** Builds many decision trees on random subsets of data and averages their predictions. Unlike linear models, trees can capture **non-linear patterns and interactions** (e.g., high volume only matters after a sharp price drop). Setting `max_depth=5` limits tree depth so trees do not memorize noisy single days.

In [ ]:
# --- LEVEL BASIC: Full Pipeline + comparison against the Naive Baseline ---
num_cols = ['return_today', 'return_5d', 'Volume']
cat_cols = ['Company']

X_train, y_train = train_df[num_cols + cat_cols], train_df['target_tomorrow_return']
X_test,  y_test  = test_df[num_cols + cat_cols],  test_df['target_tomorrow_return']

# Define separate preprocessing paths for numbers vs. text categories
preprocessor = ColumnTransformer(transformers=[
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num_cols),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='constant', fill_value='Unknown')),
                      ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_cols)
])

# 0. NAIVE BASELINE (predicts 0.0% change every day)
y_pred_naive = np.zeros(len(y_test))
results = [{
    'Model': '0. Naive Baseline (0% change)',
    'MAE (%)': round(mean_absolute_error(y_test, y_pred_naive) * 100, 3),
    'RMSE (%)': round(np.sqrt(mean_squared_error(y_test, y_pred_naive)) * 100, 3),
    'Direction Accuracy (%)': round(((y_test > 0) == (y_pred_naive >= 0)).mean() * 100, 1)
}]

# 1 & 2. Compare a linear model vs. a tree ensemble on the exact same preprocessor
models = {
    '1. Ridge (Linear)': Ridge(alpha=1.0),
    '2. Random Forest':  RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)
}

for name, model in models.items():
    pipe = Pipeline([('prep', preprocessor), ('model', model)])
    pipe.fit(X_train, y_train)        # Learns imputer, scaler, encoder, and model ONLY on train
    y_pred = pipe.predict(X_test)     # Applies saved transforms + predicts on unseen future
    results.append({
        'Model': name,
        'MAE (%)': round(mean_absolute_error(y_test, y_pred) * 100, 3),
        'RMSE (%)': round(np.sqrt(mean_squared_error(y_test, y_pred)) * 100, 3),
        'Direction Accuracy (%)': round(((y_test > 0) == (y_pred > 0)).mean() * 100, 1)
    })

display(pd.DataFrame(results))

## Step 3: What Each Metric Actually Means and How to Read Error Plots

Choosing a metric is a **product decision**, not just a math choice. Here is what each number tells you:

| Metric | Plain-English Definition | Why / When to Use It |
| :--- | :--- | :--- |
| **`MAE (%)`** *(Mean Absolute Error)* | Average of `|true - predicted|`. Every 1% error counts equally. | Easiest to explain to a user (*'our daily return forecast is off by 1.7 percentage points on average'*). |
| **`RMSE (%)`** *(Root Mean Squared Error)* | Squares each error before averaging, then takes the square root. | **Penalizes big misses disproportionately.** If missing a sudden -8% crash is far worse than small +/-0.5% misses, watch RMSE. |
| **`Direction Accuracy (%)`** | Share of test days where `sign(predicted) == sign(true)`. | Tells you whether the model at least gets the up/down direction right (often more useful for investors than exact magnitude). |

### Why do we plot Residuals (`y_true - y_pred`)?
A single average number can hide systematic bias. A **residual** is simply `actual_value - predicted_value`:
*   Positive residual -> the model **underpredicted**.
*   Negative residual -> the model **overpredicted**.
*   Ideally, residuals form a symmetric bell centered at `0`. If the bell is shifted left or right, the model is consistently biased.

**Reality check on financial data:** On synthetic random-walk data above, ML models hover near the Naive Baseline — and on real daily stock prices, beating Naive even slightly is genuinely hard. If your real dataset suddenly gives `MAE = 0.01%` or `R2 = 0.99`, stop and check for **Data Leakage**.

In [ ]:
# Visual check: histogram of residuals (y_true - y_pred)
residuals = y_test - y_pred
plt.figure(figsize=(8, 4))
sns.histplot(residuals * 100, bins=25, kde=True, color='teal')
plt.axvline(0, color='red', linestyle='--', lw=2, label='Zero error')
plt.title('Residual Distribution (y_true - y_pred)')
plt.xlabel('Prediction Error (percentage points)')
plt.legend()
plt.tight_layout()
plt.show()

---
## Step 4: Your Turn (Sprint 3 Task in Your Repository)

Create `notebooks/02_baseline.ipynb` in your repository. **Before writing code**, answer these 5 questions together in a Markdown cell at the top of your notebook:
1.  **What exact target will our app predict?** (Next-day `%` return as regression? Up/down direction as classification? Or a 5-day horizon?)
2.  **Where could Data Leakage hide in our CSV?** (Which same-day columns must be dropped or shifted?)
3.  **Are there anomalies, stock splits, or missing dates** in our dataset that could distort `pct_change()`?
4.  **What score does our Naive Baseline (0% change) achieve** on the newest 20% of dates?
5.  **Which single primary metric** (`MAE %`, `RMSE %`, or `F1` for direction) best matches our `README.md`, and why?

### Checklist for Today and Next Week
*   **BASIC:** Finish EDA (`01_eda`) on your real CSV **OR** run `notebooks/02_baseline.ipynb` with a chronological split, `groupby` lags, and a comparison table (**Naive Baseline vs. Ridge vs. RandomForest**).
*   **PRO:** Pull out the 5 test days with the largest prediction errors, check what happened to those stocks on those days, and list 3 financial indicators (e.g., moving averages, rolling volatility, volume change) you want to test next week.

In [ ]:
# =====================================================================
# STARTER SKELETON FOR YOUR notebooks/02_baseline.ipynb
# =====================================================================

# 1. Load your dataset from data/raw/ and convert the date column to datetime
# df = pd.read_csv('../data/raw/...')

# 2. Sort by [company, date], create backward-looking features + target shift(-1) using groupby
# ...

# 3. Create a CHRONOLOGICAL split (oldest 80% -> train, newest 20% -> test)
# ...

# 4. Build a ColumnTransformer + Pipeline and compare Naive Baseline vs. 2 simple models
# ...

# 5. Plot the residual distribution (or confusion matrix if you chose up/down classification)
# ...